In [217]:
import numpy as np
import pandas as pd
from joblib import load
import os

In [218]:
os.chdir(os.getcwd())

# Defining px end-members

In [219]:
# chemical format SiO2, TiO2, Al2O3, FeO, MnO, MgO, CaO, Na2O, K2O, P2O5, CO2
fs =      np.array([2,    0,   0,     0,   0,   2,   0,    0,   0,   0,   0])
en =      np.array([2,    0,   0,     2,   0,   0,   0,    0,   0,   0,   0])
wo =      np.array([2,    0,   0,     0,   0,   0,   2,    0,   0,   0,   0])

fs = 100 * fs / sum(fs)
en = 100 * en / sum(en)
wo = 100 * wo / sum(wo)

In [220]:
def make_opx(interval = 0.1):
    fs =      np.array([2,    0,   0,     0,   0,   2,   0,    0,   0,   0,   0])
    en =      np.array([2,    0,   0,     2,   0,   0,   0,    0,   0,   0,   0])
    fs = 100 * fs / sum(fs)
    en = 100 * en / sum(en)
    opx = np.array([])
    for i in np.arange(0,1 + interval,interval):
        s = round(i,2)
        comp = (fs * s) + (en * (1 - s))
        # print(comp)
        if (i == 0):
            opx = comp
        else:
            opx = np.vstack((opx,comp))
    return(opx)

def make_px(interval = 0.1):
    opx = make_opx(interval)
    wo =      np.array([2,    0,   0,     0,   0,   0,   2,    0,   0,   0,   0])
    wo = 100 * wo / sum(wo)
    c = 0
    px = np.array([])
    for i in np.arange(0,np.shape(opx)[0]):
        for j in np.arange(0,1 + interval, interval):
            comp = wo * j + (opx[i] * (1 - j))
            c = c + 1
            if (c == 1):
                px = comp
            else:
                px = np.vstack((px,comp))
    
    np.shape(px)
    return(px)

In [221]:
np.shape(make_px(0.1))

(121, 11)

In [222]:
px = make_px(0.01)
px = pd.DataFrame(px, columns = ["SiO2", "TiO2", "Al2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O", "P2O5", "CO2"])
px

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,CO2
0,50.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,50.0,0.0,0.0,49.5,0.0,0.0,0.5,0.0,0.0,0.0,0.0
2,50.0,0.0,0.0,49.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,50.0,0.0,0.0,48.5,0.0,0.0,1.5,0.0,0.0,0.0,0.0
4,50.0,0.0,0.0,48.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
10196,50.0,0.0,0.0,0.0,0.0,2.0,48.0,0.0,0.0,0.0,0.0
10197,50.0,0.0,0.0,0.0,0.0,1.5,48.5,0.0,0.0,0.0,0.0
10198,50.0,0.0,0.0,0.0,0.0,1.0,49.0,0.0,0.0,0.0,0.0
10199,50.0,0.0,0.0,0.0,0.0,0.5,49.5,0.0,0.0,0.0,0.0


# Load model (and other files) and make predictions

In [223]:
mdl = "RF"
combination = "C4"

In [224]:
if (mdl == "KNN"):
    path = r"C:\Users\naik3\Documents\Research\Mineral identifier ann IISER Mohali\Additional machine learning algos\KNN\\"
if (mdl == "SVM"):
    path = r"C:\Users\naik3\Documents\Research\Mineral identifier ann IISER Mohali\Additional machine learning algos\SVM\\"
if (mdl == "RF"):
    path = r"C:\Users\naik3\Documents\Research\Mineral identifier ann IISER Mohali\Additional machine learning algos\RF\\"

if (combination != "C4"):
    model = mdl + "_" + combination + ".mdl"
    labeler = "Labeler"+ "_" + mdl + "_" + combination + ".lbl"
    if (mdl != "RF"):
        scaler = "Scaler" + "_" + mdl + "_" + combination + ".scl"
    print(model, scaler, labeler)
else:
    model = mdl + "_" + combination + "_5_components" + ".mdl"
    labeler = "Labeler"+ "_" + mdl + "_" + combination + "_5_components" + ".lbl"
    pc = "PCA_" + mdl + "_" + combination + "_5_components" + ".pc"
    if (mdl != "RF"):
        scaler = "Scaler" + "_" + mdl + "_" + combination + "_5_components" + ".scl"
    print(model, scaler, labeler, pc)

RF_C4_5_components.mdl StandardScaler() Labeler_RF_C4_5_components.lbl PCA_RF_C4_5_components.pc


In [225]:
model = load(path + model)
if (mdl != "RF"):
    scaler = load(path + scaler)
labeler = load(path + labeler)

if (combination == "C4"):
    pc_model = load(path + pc)
    
c = px.columns

if ("PredMin" in c):
    px.pop("PredMin")
if ("Total" in c):
    px.pop("Total")

In [226]:
px1 = px.copy()

if (combination == "C2"):
    px1["M"] = px1["FeO"] + px1["MnO"] + px1["MgO"]
    px1.drop(columns=["FeO","MnO","MgO"])
    px1 = px1[["SiO2", "TiO2", "Al2O3", "M", "CaO", "Na2O", "K2O", "P2O5", "CO2"]]
    
if (combination == "C3"):
    px1["M"] = px1["FeO"] + px1["MnO"] + px1["MgO"]
    px1["A"] = px1["Na2O"] + px1["K2O"]
    px1.drop(columns=["FeO","MnO","MgO","Na2O","K2O"])
    px1 = px1[["SiO2", "TiO2", "Al2O3", "M", "CaO", "A", "P2O5", "CO2"]]

if (combination == "C4"):
    px1 = px1[["SiO2", "TiO2", "Al2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O", "P2O5", "CO2"]]
    if (mdl != "RF"):
        px1 = scaler.transform(px1)
    data_scaled = pc_model.transform(px1)

px1

C:\Users\naik3\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(


,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,CO2
0,50.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,50.0,0.0,0.0,49.5,0.0,0.0,0.5,0.0,0.0,0.0,0.0
2,50.0,0.0,0.0,49.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,50.0,0.0,0.0,48.5,0.0,0.0,1.5,0.0,0.0,0.0,0.0
4,50.0,0.0,0.0,48.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
10196,50.0,0.0,0.0,0.0,0.0,2.0,48.0,0.0,0.0,0.0,0.0
10197,50.0,0.0,0.0,0.0,0.0,1.5,48.5,0.0,0.0,0.0,0.0
10198,50.0,0.0,0.0,0.0,0.0,1.0,49.0,0.0,0.0,0.0,0.0
10199,50.0,0.0,0.0,0.0,0.0,0.5,49.5,0.0,0.0,0.0,0.0


In [227]:
if (combination != "C4"):
    if (mdl != "RF"):
        data_scaled = scaler.transform(px1)
    elif (mdl == "RF"):
        data_scaled = px1.copy()

In [228]:
data_scaled

array([[ -0.38362144,  -8.09335904,   4.27304875, -31.38139242,
         32.32988269],
       [ -0.41400463,  -7.99787157,   4.71054385, -30.92437934,
         32.06548646],
       [ -0.44438781,  -7.9023841 ,   5.14803895, -30.46736625,
         31.80109022],
       ...,
       [ -3.93259684,   1.14096526,  46.83837875,  14.25186768,
          5.50063859],
       [ -3.67726843,   1.29817662,  47.43046873,  14.28589191,
          5.69544871],
       [ -3.42194001,   1.45538798,  48.02255871,  14.31991613,
          5.89025884]])

In [229]:
pred = model.predict(data_scaled)
predictions = labeler.inverse_transform(pred)

In [230]:
predictions[predictions != "Px"]

array(['Amp', 'Amp', 'Amp', ..., 'Grt', 'Grt', 'Grt'], dtype=object)

In [231]:
x = model.predict_proba(data_scaled)
px["Total"] = px.sum(axis = 1)
px["PredMin"] = predictions
px_prob = pd.DataFrame(x, columns = labeler.classes_, index = px.index)

In [232]:
px_final = pd.concat([px,px_prob], axis = 1)

In [233]:
px_final

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,...,Ilm,Mag/Hem,Ms,Ol,Px,Qz,Rt,Spl,St,Ttn
0,50.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.02,0.82,0.0,0.0,0.0,0.0,0.0
1,50.0,0.0,0.0,49.5,0.0,0.0,0.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.02,0.86,0.0,0.0,0.0,0.0,0.0
2,50.0,0.0,0.0,49.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.02,0.88,0.0,0.0,0.0,0.0,0.0
3,50.0,0.0,0.0,48.5,0.0,0.0,1.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.90,0.0,0.0,0.0,0.0,0.0
4,50.0,0.0,0.0,48.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.84,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10196,50.0,0.0,0.0,0.0,0.0,2.0,48.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.22,0.0,0.0,0.0,0.0,0.0
10197,50.0,0.0,0.0,0.0,0.0,1.5,48.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.22,0.0,0.0,0.0,0.0,0.0
10198,50.0,0.0,0.0,0.0,0.0,1.0,49.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.22,0.0,0.0,0.0,0.0,0.0
10199,50.0,0.0,0.0,0.0,0.0,0.5,49.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.22,0.0,0.0,0.0,0.0,0.0


In [234]:
px_final.Total

0        100.0
1        100.0
2        100.0
3        100.0
4        100.0
         ...  
10196    100.0
10197    100.0
10198    100.0
10199    100.0
10200    100.0
Name: Total, Length: 10201, dtype: float64

In [235]:
output_file = "Px_prediction_for_" + mdl + "_" + combination + ".csv"
output_file

'Px_prediction_for_RF_C4.csv'

In [236]:
px_final.to_csv(output_file)